# Practica M33: Análisis exploratorio de ventas de videojuegos (EDA)
#### Objetivo: explorar las ventas históricas de videojuegos para encontrar información útil para una posible nueva plataforma.

## 1. Importar librerías y leer el archivo

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Execute from the repository root (or the video-game-eda directory).
working_dir = Path.cwd()
if (working_dir / "data-analysis" / "video-game-eda").is_dir():
    project_dir = working_dir / "data-analysis" / "video-game-eda"
elif (working_dir / "video-game-eda").is_dir():
    project_dir = working_dir / "video-game-eda"
else:
    project_dir = working_dir

descargas = project_dir / "generated"
descargas.mkdir(parents=True, exist_ok=True)
archivo = project_dir / "vgsales.csv"
carpeta_graficas = descargas / "figures"
carpeta_graficas.mkdir(parents=True, exist_ok=True)

# The class-provided dataset is public but is not redistributed in the repo.
# Download it with gdown, or manually place vgsales.csv next to this README.
if not archivo.exists():
    try:
        import gdown
        gdown.download(id="1YNDD-qJkSiz3EN4jGD9pOC-kEeBMURmf", output=str(archivo), quiet=False)
    except ImportError:
        print("Install gdown (pip install gdown) or download vgsales.csv manually.")

if not archivo.exists():
    raise FileNotFoundError(
        f"Missing {archivo}. Download the source dataset from the course link."
    )

df = pd.read_csv(archivo)
print("Archivo cargado:", archivo)
print("Filas:", len(df), "| Columnas:", len(df.columns))
print("Columnas:", df.columns.tolist())


## 2. Exploración inicial: head, tail, sample, describe y tipos

In [ ]:
print("PRIMERAS 5 FILAS")
display(df.head())
print("ÚLTIMAS 5 FILAS")
display(df.tail())
print("MUESTRA ALEATORIA")
display(df.sample(min(10, len(df)), random_state=42))

In [ ]:
print("TIPOS DE DATOS")
print(df.dtypes)
print("VALORES VACÍOS POR COLUMNA")
print(df.isna().sum())
print("ESTADÍSTICAS ANTES DE LA CONVERSIÓN")
display(df.describe(include="all"))

## 3. Limpieza básica y validación de columnas numéricas
#### Como vimos en clase, pd.to_numeric() transforma las columnas de ventas y el año en números. Conservamos una copia original y para los gráficos que dependen de Year quitamos únicamente las filas sin año válido. No se inventan valores para los datos faltantes.

In [ ]:
columnas_ventas = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]
columnas_numericas = ["Rank", "Year"] + columnas_ventas

# Convierto estas columnas a números por si hay valores guardados como texto.
for columna in columnas_numericas:
    df[columna] = pd.to_numeric(df[columna], errors="coerce")

print("ESTADÍSTICAS NUMÉRICAS DESPUÉS DE LA CONVERSIÓN")
display(df[columnas_numericas].describe())

# Evito filas sin datos indispensables y años que estén fuera de rango.
df_limpio = df.dropna(subset=["Name", "Platform", "Genre"] + columnas_ventas).copy()
df_anios = df_limpio.dropna(subset=["Year"]).copy()
df_anios = df_anios[(df_anios["Year"] >= 1980) & (df_anios["Year"] <= 2026)].copy()
df_anios["Year"] = df_anios["Year"].astype(int)

print("Registros originales:", len(df))
print("Registros válidos para el análisis general:", len(df_limpio))
print("Registros con año válido:", len(df_anios))
print("Filas sin año válido:", len(df_limpio) - len(df_anios))

## 4. Totales del mercado analizado

In [ ]:
# Cada fila es un registro de videojuego en una plataforma; un título puede aparecer varias veces.
resumen_totales = pd.DataFrame({
    "Indicador": ["Registros juego-plataforma", "Videojuegos únicos", "Géneros", "Plataformas"],
    "Total": [len(df_limpio), df_limpio["Name"].nunique(),
              df_limpio["Genre"].nunique(), df_limpio["Platform"].nunique()]
})
display(resumen_totales)
resumen_totales.to_csv(descargas / "vgsales_resumen_totales.csv", index=False)

## 5. Seleccionar los últimos cuatro años comparables
#### Algunas versiones de vgsales.csv contienen uno o pocos registros de años posteriores a 2016. Para no interpretar esos años incompletos como un colapso real del mercado, se eligen los últimos cuatro años que tengan al menos 100 registros. Esta es una regla práctica, no una garantía de cobertura completa.

In [ ]:
conteo_por_anio = df_anios.groupby("Year").size()
anios_comparables = conteo_por_anio[conteo_por_anio >= 100].index.tolist()
ultimos_4 = sorted(anios_comparables)[-4:]

if len(ultimos_4) < 4:
    raise ValueError("No hay cuatro años con al menos 100 registros para comparar.")

reciente = df_anios[df_anios["Year"].isin(ultimos_4)].copy()
anio_inicio = min(ultimos_4)
anio_fin = max(ultimos_4)

print("Años elegidos:", ultimos_4)
print("Cantidad de registros por año:")
display(conteo_por_anio.loc[ultimos_4].rename("Juegos").to_frame())

## 6. Ventas por año, desglosadas por región (gráfico 01)

In [ ]:
ventas_por_anio = df_anios.groupby("Year")[columnas_ventas].sum().reset_index()
display(ventas_por_anio.tail(12).round(2))
ventas_por_anio.to_csv(descargas / "vgsales_ventas_por_anio.csv", index=False)

plt.figure(figsize=(12, 5))
for region in columnas_ventas:
    plt.plot(ventas_por_anio["Year"], ventas_por_anio[region], marker=".", label=region)
plt.title("Ventas por año de lanzamiento y región")
plt.xlabel("Año de lanzamiento")
plt.ylabel("Millones de copias")
plt.legend()
plt.tight_layout()
plt.savefig(carpeta_graficas / "01_ventas_por_anio.png", dpi=150)
plt.show()

## 7. Participación de las regiones en las ventas (gráfico 02)

In [ ]:
regiones = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
ventas_region = df_anios[regiones].sum().sort_values(ascending=False)
participacion_region = (ventas_region / ventas_region.sum() * 100).round(2)
display(participacion_region.rename("Porcentaje de ventas"))
participacion_region.to_csv(descargas / "vgsales_participacion_regiones.csv")

plt.figure(figsize=(8, 4))
sns.barplot(x=participacion_region.values, y=participacion_region.index, color="steelblue")
plt.title("Participación regional en ventas registradas")
plt.xlabel("Porcentaje del total regional (%)")
plt.ylabel("Región")
plt.tight_layout()
plt.savefig(carpeta_graficas / "02_participacion_regiones.png", dpi=150)
plt.show()

## 8. Juegos más vendidos por cada año y cada región
#### Sumo las ventas de un mismo título cuando aparece en varias plataformas durante el mismo año. Así puedo identificar el líder de NA, EU, JP, Otros y Global en cada año. Los resultados se guardan en CSV.

In [ ]:
ventas_juego_anio = df_anios.groupby(["Year", "Name"], as_index=False)[columnas_ventas].sum()
mejores_juegos = []

for region in columnas_ventas:
    indices = ventas_juego_anio.groupby("Year")[region].idxmax()
    tabla = ventas_juego_anio.loc[indices, ["Year", "Name", region]].copy()
    tabla.columns = ["Year", "Juego", "Ventas_millones"]
    tabla["Region"] = region
    mejores_juegos.append(tabla)

mejores_juegos = pd.concat(mejores_juegos, ignore_index=True)
mejores_juegos = mejores_juegos.sort_values(["Year", "Region"])
display(mejores_juegos[mejores_juegos["Year"].isin(ultimos_4)].head(25))
mejores_juegos.to_csv(descargas / "vgsales_mejor_juego_por_anio_y_region.csv", index=False)

## 9. Juegos más vendidos del periodo reciente (gráfico 03)

In [ ]:
top_juegos = (reciente.groupby("Name")["Global_Sales"].sum()
              .sort_values(ascending=False).head(10).sort_values())
plt.figure(figsize=(10, 5))
top_juegos.plot(kind="barh", color="teal")
plt.title(f"Juegos más vendidos entre {anio_inicio} y {anio_fin} (por año de lanzamiento)")
plt.xlabel("Millones de copias globales")
plt.ylabel("Videojuego")
plt.tight_layout()
plt.savefig(carpeta_graficas / "03_top_juegos.png", dpi=150)
plt.show()

## 10. Plataforma líder por año y región

In [ ]:
ventas_plataforma_anio = df_anios.groupby(["Year", "Platform"], as_index=False)[columnas_ventas].sum()
mejores_plataformas = []

for region in columnas_ventas:
    indices = ventas_plataforma_anio.groupby("Year")[region].idxmax()
    tabla = ventas_plataforma_anio.loc[indices, ["Year", "Platform", region]].copy()
    tabla.columns = ["Year", "Plataforma", "Ventas_millones"]
    tabla["Region"] = region
    mejores_plataformas.append(tabla)

mejores_plataformas = pd.concat(mejores_plataformas, ignore_index=True)
mejores_plataformas = mejores_plataformas.sort_values(["Year", "Region"])
display(mejores_plataformas[mejores_plataformas["Year"].isin(ultimos_4)].head(25))
mejores_plataformas.to_csv(descargas / "vgsales_mejor_plataforma_por_anio_y_region.csv", index=False)

## 11. Plataformas más vendidas en el periodo reciente (gráfico 04)

In [ ]:
top_plataformas = (reciente.groupby("Platform")["Global_Sales"].sum()
                   .sort_values(ascending=False).head(10).sort_values())
plt.figure(figsize=(10, 5))
top_plataformas.plot(kind="barh", color="royalblue")
plt.title(f"Plataformas con mayor volumen de ventas: {anio_inicio}-{anio_fin}")
plt.xlabel("Millones de copias globales")
plt.ylabel("Plataforma")
plt.tight_layout()
plt.savefig(carpeta_graficas / "04_top_plataformas.png", dpi=150)
plt.show()

## 12. Comparación de ventas entre géneros por año (gráfico 05)

In [ ]:
ventas_genero_anio = (df_anios.groupby(["Year", "Genre"])[columnas_ventas]
                     .sum().reset_index())
ventas_genero_anio.to_csv(descargas / "vgsales_generos_por_anio_y_region.csv", index=False)

# Para que la gráfica sea legible, muestro los últimos 15 años con cobertura suficiente.
anios_grafico = sorted(anios_comparables)[-15:]
matriz_generos = ventas_genero_anio[ventas_genero_anio["Year"].isin(anios_grafico)]
matriz_generos = matriz_generos.pivot(index="Genre", columns="Year", values="Global_Sales").fillna(0)

plt.figure(figsize=(13, 6))
sns.heatmap(matriz_generos, cmap="YlGnBu")
plt.title("Ventas globales por género y año de lanzamiento")
plt.xlabel("Año de lanzamiento")
plt.ylabel("Género")
plt.tight_layout()
plt.savefig(carpeta_graficas / "05_ventas_genero_anio.png", dpi=150)
plt.show()

## 13. Insight 1: promedio de ventas por registro y género (gráfico 06)
#### Además de las ventas totales, comparo el promedio por juego-plataforma. Un género con muchas ventas puede deberse a que tiene muchos títulos; el promedio permite evaluar otra perspectiva. Solo muestro géneros con 10 o más registros en el periodo.

In [ ]:
rendimiento_generos = (reciente.groupby("Genre")
                       .agg(Registros=("Name", "count"),
                            Ventas_globales=("Global_Sales", "sum"),
                            Promedio_por_registro=("Global_Sales", "mean"))
                       .reset_index())
rendimiento_generos.to_csv(descargas / "vgsales_rendimiento_generos.csv", index=False)

para_grafico = rendimiento_generos[rendimiento_generos["Registros"] >= 10]
para_grafico = para_grafico.sort_values("Promedio_por_registro")
plt.figure(figsize=(10, 5))
sns.barplot(data=para_grafico, x="Promedio_por_registro", y="Genre", color="seagreen")
plt.title("Venta promedio por registro y género (periodo reciente)")
plt.xlabel("Millones de copias por registro")
plt.ylabel("Género")
plt.tight_layout()
plt.savefig(carpeta_graficas / "06_rendimiento_genero.png", dpi=150)
plt.show()

## 14. Crecimiento de géneros en los últimos cuatro años (gráfico 07)
#### Comparo las ventas acumuladas de juegos lanzados en el primer y último año del periodo. No representa crecimiento real de las ventas anuales de un mismo producto. Para calcular porcentaje, el valor inicial debe ser mayor que cero; cuando no había registros al inicio, el porcentaje queda vacío y puede tratarse como una categoría nueva.

In [ ]:
def tabla_crecimiento(columna_grupo):
    # Agrupo los datos por año y categoría y sumo ventas de las 5 regiones.
    base = reciente.groupby([columna_grupo, "Year"])[columnas_ventas].sum().reset_index()
    resultado = base[base["Year"] == anio_inicio].drop(columns="Year")
    final = base[base["Year"] == anio_fin].drop(columns="Year")
    resultado = resultado.merge(final, on=columna_grupo, how="outer", suffixes=("_inicio", "_fin")).fillna(0)
    for region in columnas_ventas:
        resultado[region + "_cambio"] = resultado[region + "_fin"] - resultado[region + "_inicio"]
        resultado[region + "_pct"] = (resultado[region + "_cambio"] /
                                      resultado[region + "_inicio"].replace(0, np.nan)) * 100
    return resultado

crecimiento_generos = tabla_crecimiento("Genre")
crecimiento_generos.to_csv(descargas / "vgsales_crecimiento_generos_regiones.csv", index=False)

# Comparo solamente categorías con ventas en los dos años.
generos_comparables = crecimiento_generos[(crecimiento_generos["Global_Sales_inicio"] > 0) &
                                         (crecimiento_generos["Global_Sales_fin"] > 0)]
generos_ordenados = generos_comparables.sort_values("Global_Sales_pct")

print("Género con mayor crecimiento porcentual:")
display(generos_ordenados.tail(1)[["Genre", "Global_Sales_pct"]])
print("Género con menor crecimiento porcentual:")
display(generos_ordenados.head(1)[["Genre", "Global_Sales_pct"]])

plt.figure(figsize=(10, 5))
sns.barplot(data=generos_ordenados, x="Global_Sales_pct", y="Genre", color="cadetblue")
plt.axvline(0, color="black", linewidth=1)
plt.title(f"Variación de ventas por género: {anio_inicio} vs. {anio_fin}")
plt.xlabel("Cambio porcentual (%)")
plt.ylabel("Género")
plt.tight_layout()
plt.savefig(carpeta_graficas / "07_crecimiento_generos.png", dpi=150)
plt.show()

## 15. Crecimiento de plataformas (gráfico 08)

In [ ]:
crecimiento_plataformas = tabla_crecimiento("Platform")
crecimiento_plataformas.to_csv(descargas / "vgsales_crecimiento_plataformas_regiones.csv", index=False)

plataformas_comparables = crecimiento_plataformas[(crecimiento_plataformas["Global_Sales_inicio"] > 0) &
                                                 (crecimiento_plataformas["Global_Sales_fin"] > 0)]
plataformas_ordenadas = plataformas_comparables.sort_values("Global_Sales_pct")
print("Plataforma con mayor crecimiento porcentual:")
display(plataformas_ordenadas.tail(1)[["Platform", "Global_Sales_pct"]])
print("Plataforma con menor crecimiento porcentual:")
display(plataformas_ordenadas.head(1)[["Platform", "Global_Sales_pct"]])

plt.figure(figsize=(10, 5))
sns.barplot(data=plataformas_ordenadas, x="Global_Sales_pct", y="Platform", color="steelblue")
plt.axvline(0, color="black", linewidth=1)
plt.title(f"Variación de ventas por plataforma: {anio_inicio} vs. {anio_fin}")
plt.xlabel("Cambio porcentual (%)")
plt.ylabel("Plataforma")
plt.tight_layout()
plt.savefig(carpeta_graficas / "08_crecimiento_plataformas.png", dpi=150)
plt.show()

## 16. Crecimiento de videojuegos: comparación aproximada (gráfico 09)
#### Límite del archivo: cada registro tiene ventas totales y año de lanzamiento; no tenemos las ventas de cada videojuego año por año. Para realizar el ejercicio, comparo los títulos que aparecen en al menos dos años distintos dentro del periodo. Esto suele representar ediciones o lanzamientos en otras plataformas, por lo que es un indicador aproximado, no crecimiento comercial del mismo juego.

In [ ]:
ventas_nombre_anio = (reciente.groupby(["Name", "Year"], as_index=False)[columnas_ventas].sum()
                     .sort_values(["Name", "Year"]))
comparaciones_juegos = []
for nombre, grupo in ventas_nombre_anio.groupby("Name"):
    if grupo["Year"].nunique() >= 2:
        primero = grupo.iloc[0]
        ultimo = grupo.iloc[-1]
        inicial = primero["Global_Sales"]
        final = ultimo["Global_Sales"]
        if inicial > 0:
            comparaciones_juegos.append({
                "Juego": nombre, "Anio_inicial": int(primero["Year"]),
                "Anio_final": int(ultimo["Year"]), "Ventas_iniciales": inicial,
                "Ventas_finales": final, "Cambio_millones": final - inicial,
                "Cambio_pct": (final - inicial) / inicial * 100
            })

crecimiento_juegos = pd.DataFrame(comparaciones_juegos)
crecimiento_juegos.to_csv(descargas / "vgsales_variacion_juegos_aproximada.csv", index=False)

if len(crecimiento_juegos) > 0:
    orden_juegos = crecimiento_juegos.sort_values("Cambio_millones")
    print("Mayor variación absoluta:")
    display(orden_juegos.tail(1))
    print("Menor variación absoluta:")
    display(orden_juegos.head(1))
    grafico_juegos = pd.concat([orden_juegos.head(5), orden_juegos.tail(5)])
    grafico_juegos = grafico_juegos.sort_values("Cambio_millones")
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grafico_juegos, x="Cambio_millones", y="Juego", color="mediumpurple")
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Variación aproximada entre lanzamientos del mismo nombre")
    plt.xlabel("Diferencia de millones de copias")
    plt.ylabel("Título")
    plt.tight_layout()
    plt.savefig(carpeta_graficas / "09_variacion_videojuegos.png", dpi=150)
    plt.show()
else:
    print("No hay títulos repetidos en años distintos: no es posible estimar esta variación.")

## 17. Oportunidad de mercado 1: preferencias de género por región (gráfico 10)
#### La proporción de cada género dentro de cada mercado regional puede mostrar diferencias de interés. Son señales exploratorias para decidir qué tipo de catálogo analizar mejor, no una prueba de demanda futura.

In [ ]:
ventas_genero_region = reciente.groupby("Genre")[regiones].sum()
# Divido cada columna entre el total de su región para comparar porcentajes.
porcentaje_genero_region = ventas_genero_region.div(ventas_genero_region.sum(axis=0), axis=1) * 100
porcentaje_genero_region.to_csv(descargas / "vgsales_oportunidad_genero_region.csv")

plt.figure(figsize=(10, 6))
sns.heatmap(porcentaje_genero_region, annot=True, fmt=".1f", cmap="YlOrBr")
plt.title("Porcentaje de ventas de cada región por género")
plt.xlabel("Región")
plt.ylabel("Género")
plt.tight_layout()
plt.savefig(carpeta_graficas / "10_oportunidad_genero_region.png", dpi=150)
plt.show()

## 18. Oportunidad de mercado 2: oferta vs. ventas por juego (gráfico 11)
#### Comparo el número de registros de cada género (una aproximación a la oferta) contra su venta promedio. Un género con pocos títulos pero buen promedio podría merecer una investigación adicional; también puede ser resultado de un solo éxito.

In [ ]:
oportunidad_generos = rendimiento_generos[rendimiento_generos["Registros"] >= 5].copy()
plt.figure(figsize=(9, 5))
sns.scatterplot(data=oportunidad_generos, x="Registros", y="Promedio_por_registro",
                size="Ventas_globales", sizes=(70, 450), legend=False, color="darkcyan")
for _, fila in oportunidad_generos.iterrows():
    plt.annotate(fila["Genre"], (fila["Registros"], fila["Promedio_por_registro"]),
                 xytext=(4, 3), textcoords="offset points", fontsize=8)
plt.title("Oportunidad exploratoria: oferta vs. venta promedio por género")
plt.xlabel("Número de registros")
plt.ylabel("Millones de copias por registro")
plt.tight_layout()
plt.savefig(carpeta_graficas / "11_oportunidad_oferta_ventas.png", dpi=150)
plt.show()

## 19. Resumen automático para la presentación
#### Esta celda imprime los hallazgos principales y genera vgsales_resumen_para_presentacion.txt en Downloads. Usare estos datos para generar el desarrollo de las diapositivas.

In [ ]:
# El archivo de resumen ayuda a completar los textos del PowerPoint sin inventar datos.
lineas = [
    "EDA DE VENTAS DE VIDEOJUEGOS",
    f"Registros válidos: {len(df_limpio):,}",
    f"Títulos únicos: {df_limpio['Name'].nunique():,}",
    f"Géneros: {df_limpio['Genre'].nunique()} | Plataformas: {df_limpio['Platform'].nunique()}",
    f"Últimos 4 años comparables: {ultimos_4}",
    "Medida: millones de copias acumuladas por año de LANZAMIENTO, no ingresos monetarios.",
    f"Región de mayor participación: {participacion_region.idxmax()} ({participacion_region.max():.1f}%)",
]

if len(generos_ordenados):
    lineas += [f"Género con mayor variación porcentual: {generos_ordenados.iloc[-1]['Genre']} ({generos_ordenados.iloc[-1]['Global_Sales_pct']:.1f}%)",
               f"Género con menor variación porcentual: {generos_ordenados.iloc[0]['Genre']} ({generos_ordenados.iloc[0]['Global_Sales_pct']:.1f}%)"]
if len(plataformas_ordenadas):
    lineas += [f"Plataforma con mayor variación porcentual: {plataformas_ordenadas.iloc[-1]['Platform']} ({plataformas_ordenadas.iloc[-1]['Global_Sales_pct']:.1f}%)",
               f"Plataforma con menor variación porcentual: {plataformas_ordenadas.iloc[0]['Platform']} ({plataformas_ordenadas.iloc[0]['Global_Sales_pct']:.1f}%)"]
if len(crecimiento_juegos):
    lineas += [f"Título con mayor cambio aproximado: {orden_juegos.iloc[-1]['Juego']} ({orden_juegos.iloc[-1]['Cambio_millones']:.2f} millones)",
               f"Título con menor cambio aproximado: {orden_juegos.iloc[0]['Juego']} ({orden_juegos.iloc[0]['Cambio_millones']:.2f} millones)"]

resumen_texto = "\n".join(lineas)
print(resumen_texto)
(descargas / "vgsales_resumen_para_presentacion.txt").write_text(resumen_texto, encoding="utf-8")
print("Gráficos disponibles en:", carpeta_graficas)